# OpenOppsDB skills radar

This topical notebook explores skill groups, keywords, role slices, and skill-pair co-occurrence in current open roles.


In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")


In [ ]:
with sqlite3.connect(DB_URI, uri=True) as conn:
    top_skills = pd.read_sql_query(
        """
        select lower(s.name) as skill, count(distinct j.id) as open_roles
        from jobs j
        join job_versions v on v.id = j.current_version_id
        join job_version_skills s on s.job_version_id = v.id
        where j.status = 'open' and s.name is not null and s.name <> ''
        group by lower(s.name)
        order by open_roles desc, skill
        limit 25
        """,
        conn,
    )
    top_keywords = pd.read_sql_query(
        """
        select lower(k.keyword) as keyword, count(distinct j.id) as open_roles
        from jobs j
        join job_versions v on v.id = j.current_version_id
        join job_version_skills s on s.job_version_id = v.id
        join job_version_skill_keywords k on k.skill_id = s.id
        where j.status = 'open' and k.keyword is not null and k.keyword <> ''
        group by lower(k.keyword)
        order by open_roles desc, keyword
        limit 25
        """,
        conn,
    )
    skill_pairs = pd.read_sql_query(
        """
        select
            lower(s1.name) as skill_a,
            lower(s2.name) as skill_b,
            count(distinct j.id) as open_roles
        from jobs j
        join job_versions v on v.id = j.current_version_id
        join job_version_skills s1 on s1.job_version_id = v.id
        join job_version_skills s2
          on s2.job_version_id = v.id and lower(s1.name) < lower(s2.name)
        where j.status = 'open'
          and s1.name is not null and s1.name <> ''
          and s2.name is not null and s2.name <> ''
        group by lower(s1.name), lower(s2.name)
        order by open_roles desc, skill_a, skill_b
        limit 25
        """,
        conn,
    )

display(top_skills)
display(top_keywords)
skill_pairs


In [ ]:
with sqlite3.connect(DB_URI, uri=True) as conn:
    role_slices = pd.read_sql_query(
        """
        with role_labels as (
            select
                v.id as version_id,
                case
                    when lower(v.title) like '%data%' then 'data'
                    when lower(v.title) like '%engineer%' then 'engineering'
                    when lower(v.title) like '%product%' then 'product'
                    when lower(v.title) like '%sales%' then 'sales'
                    else 'other'
                end as role_slice
            from jobs j
            join job_versions v on v.id = j.current_version_id
            where j.status = 'open'
        )
        select
            r.role_slice,
            lower(s.name) as skill,
            count(*) as mentions
        from role_labels r
        join job_version_skills s on s.job_version_id = r.version_id
        where s.name is not null and s.name <> ''
        group by r.role_slice, lower(s.name)
        order by mentions desc, role_slice, skill
        limit 40
        """,
        conn,
    )

role_slices


In [ ]:
if not top_skills.empty:
    ax = top_skills.head(15).sort_values("open_roles").plot.barh(
        x="skill", y="open_roles", figsize=(10, 6), legend=False, title="Top skills"
    )
    ax.set_xlabel("Current open roles")
    plt.tight_layout()
    plt.show()
else:
    print("No skill rows found in this snapshot.")
